# Structured output: schema + validate + repair

**Session 4 · small model (`llama3.2:3b`) vs big model (`gpt-oss:120b-cloud`)**

Force JSON, validate with pydantic, repair on failure — then measure, over a real set, how
often the first attempt is valid. A vague "give me JSON" prompt fails almost every time (it
returns markdown or prose); a strict prompt that spells out the exact schema succeeds almost
every time. The repair loop is what saves the vague case — but the cheaper fix is the prompt.

In [ ]:
import sys; sys.path.append('..')  # so `utils` and `eval` import from the repo root
import json
from pydantic import BaseModel, ValidationError, field_validator
from utils import ask, SMALL_MODEL, BIG_MODEL
from eval import load_cases


In [ ]:
# A schema with an enum, a bounded float, and a nested list of objects. The strict prompt
# below hits it every time on the 3B model; the vague prompt essentially never does.
class Aspect(BaseModel):
    name: str
    sentiment: str

    @field_validator("sentiment")
    @classmethod
    def _known(cls, v):
        if v not in {"positive", "negative", "neutral"}:
            raise ValueError("sentiment must be positive|negative|neutral")
        return v

class Review(BaseModel):
    label: str
    confidence: float
    aspects: list[Aspect]

    @field_validator("label")
    @classmethod
    def _label(cls, v):
        if v not in {"positive", "negative", "neutral"}:
            raise ValueError("label must be positive|negative|neutral")
        return v

    @field_validator("confidence")
    @classmethod
    def _conf(cls, v):
        if not 0.0 <= v <= 1.0:
            raise ValueError("confidence must be between 0 and 1")
        return v

def parse(raw):
    raw = raw[raw.find("{"): raw.rfind("}") + 1]
    return Review(**json.loads(raw))

print(parse('{"label":"positive","confidence":0.9,"aspects":[{"name":"battery","sentiment":"positive"}]}'))


### The extract-with-repair loop

`extract()` returns `(review, n_repairs)`. We'll run it over 20 reviews with a **loose**
prompt and a **strict** prompt and compare first-try and post-repair success rates.

In [ ]:
LOOSE = 'Give me JSON describing the sentiment of: "{t}"'
STRICT = (
    'Return ONLY a JSON object, no prose, no code fence, matching exactly:\n'
    '{{"label": "positive|negative|neutral", "confidence": <float 0-1>, '
    '"aspects": [{{"name": "<phrase>", "sentiment": "positive|negative|neutral"}}]}}\n'
    'Text: "{t}"'
)

def extract(text, prompt_tmpl, model, max_repairs=2):
    raw = ask(prompt_tmpl.format(t=text), model=model)
    for n in range(max_repairs + 1):
        try:
            return parse(raw), n
        except (json.JSONDecodeError, ValidationError, ValueError) as e:
            raw = ask(
                f'Your last output was invalid: {e}\n'
                f'Return ONLY valid JSON matching:\n{STRICT.format(t=text)}',
                model=model,
            )
    return None, max_repairs + 1

reviews = [c["input"] for c in load_cases("../eval/datasets/sentiment.jsonl")][:20]

def report(prompt_tmpl, model, name):
    first_ok = solved = repairs = 0
    for t in reviews:
        res, n = extract(t, prompt_tmpl, model)
        first_ok += (res is not None and n == 0)
        solved += res is not None
        repairs += n
    print(f"  {name:22} first-try {first_ok}/{len(reviews)}   "
          f"valid after repair {solved}/{len(reviews)}   "
          f"{repairs} repair calls total")

print(f"small model ({SMALL_MODEL}):")
report(LOOSE, SMALL_MODEL, "loose prompt")
report(STRICT, SMALL_MODEL, "strict prompt")


### Same test on the big model

The repair loop is a safety net, not a plan. A bigger model needs fewer repair calls for the
same schema — that saved latency is part of the model's price/performance.

In [ ]:
try:
    print(f"big model ({BIG_MODEL}):")
    report(LOOSE, BIG_MODEL, "loose prompt")
    report(STRICT, BIG_MODEL, "strict prompt")
except Exception as e:
    print(f"skipped big-model run: {type(e).__name__}: {e}")


## Your turn - vary the example

1. Add another field (e.g. `summary: str`) and update both the schema string and the model.
2. Make it fail a different way (ask for YAML, or a wrong enum) and confirm repair recovers.
3. Log how many retries each input needs. What is your acceptable retry budget?
